In [ ]:
-- =============================================================================
-- CMS Pipeline: Bronze -> Silver -> Gold
-- FIX 2026-07-28: DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE
--   CS-01..04: CROSS JOIN latest_batch -> chi lay MAX(crawled_at)
--   CS-03 news: them guard AND (tgt.updatedAt < src.updatedAt OR tgt.updatedAt IS NULL)
--   CG-03 destination: hierarchy CTE them ROW_NUMBER() -> chi lay 1 parent / dest_id
-- =============================================================================


-- =============================================================================
-- CB-00  Tao Bronze Table (DDL)
-- =============================================================================
CREATE TABLE IF NOT EXISTS lh_vv_bronze.dbo.cms_raw_data (
    event_id        STRING      NOT NULL    COMMENT 'Unique ingest record id.',
    source_name     STRING      NOT NULL    COMMENT 'Source system name.',
    type            STRING      NOT NULL    COMMENT 'Content type.',
    crawled_at      STRING      NOT NULL    COMMENT 'When source was crawled.',
    ingested_date   STRING      NOT NULL    COMMENT 'Date partition key derived from ingested_at.',
    raw_payload     STRING      NOT NULL    COMMENT 'Original source JSON payload as STRING for replay.'
)
USING DELTA
PARTITIONED BY (ingested_date, source_name)
COMMENT 'Bronze raw CMS / content data landing table. Immutable append-only.'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite'   = 'true',
    'delta.autoOptimize.autoCompact'     = 'true',
    'delta.logRetentionDuration'         = 'interval 90 days',
    'delta.deletedFileRetentionDuration' = 'interval 14 days',
    'delta.enableChangeDataFeed'         = 'true',
    'layer'                              = 'bronze',
    'domain'                             = 'content'
);


-- =============================================================================
-- CS-00  Tao Silver Tables (DDL)
-- =============================================================================
CREATE TABLE IF NOT EXISTS lh_vv_silver.dbo.cms_destination (
    language        STRING      NOT NULL COMMENT 'Language code (locale) from CMS',
    id              STRING      NOT NULL COMMENT 'CMS destination identifier',
    name            STRING      NOT NULL COMMENT 'Destination display name',
    name_normalized STRING               COMMENT 'Normalized name / slug',
    description     STRING               COMMENT 'Editorial description text',
    documentId      STRING               COMMENT 'Strapi document identifier',
    createdAt       TIMESTAMP            COMMENT 'Record creation timestamp in CMS',
    updatedAt       TIMESTAMP            COMMENT 'Last update timestamp in CMS',
    publishedAt     TIMESTAMP            COMMENT 'Publication timestamp',
    _created_at     TIMESTAMP   NOT NULL COMMENT 'Ingestion timestamp',
    _updated_at     TIMESTAMP   NOT NULL COMMENT 'Last update timestamp'
)
USING DELTA
PARTITIONED BY (language)
COMMENT 'Silver layer CMS destination content'
TBLPROPERTIES ('delta.enableChangeDataFeed'='true','layer'='silver','domain'='content');

CREATE TABLE IF NOT EXISTS lh_vv_silver.dbo.cms_blog (
    locale                      STRING      NOT NULL COMMENT 'Language code (locale)',
    id                          STRING      NOT NULL COMMENT 'Strapi numeric id cast to STRING',
    document_id                 STRING               COMMENT 'Strapi documentId (globally unique)',
    slug                        STRING,
    title                       STRING,
    sapo                        STRING,
    author                      STRING,
    block_content_json          STRING,
    plain_text_body             STRING,
    cover_image_url             STRING,
    cover_image_medium_url      STRING,
    cover_image_small_url       STRING,
    cover_image_thumbnail_url   STRING,
    gallery_urls                STRING,
    destination_slug            STRING,
    poi_ids                     STRING,
    platforms                   STRING,
    seo_title                   STRING,
    seo_meta_description        STRING,
    first_publish_date          DATE,
    publishedAt                 TIMESTAMP,
    createdAt                   TIMESTAMP,
    updatedAt                   TIMESTAMP,
    _created_at                 TIMESTAMP,
    _updated_at                 TIMESTAMP
)
USING DELTA
COMMENT 'Silver CMS blog'
TBLPROPERTIES (
    'delta.enableChangeDataFeed'        = 'true',
    'delta.autoOptimize.optimizeWrite'  = 'true',
    'delta.autoOptimize.autoCompact'    = 'true',
    'layer'                             = 'silver',
    'domain'                            = 'content'
);

CREATE TABLE IF NOT EXISTS lh_vv_silver.dbo.cms_news (
    locale                      STRING      NOT NULL COMMENT 'Language code (locale)',
    id                          STRING      NOT NULL COMMENT 'Strapi numeric id cast to STRING',
    document_id                 STRING,
    slug                        STRING,
    title                       STRING,
    sapo                        STRING,
    is_featured                 BOOLEAN,
    block_content_json          STRING,
    plain_text_body             STRING,
    cover_image_url             STRING,
    cover_image_medium_url      STRING,
    cover_image_small_url       STRING,
    cover_image_thumbnail_url   STRING,
    news_category_key           STRING,
    news_category_name          STRING,
    seo_title                   STRING,
    seo_meta_description        STRING,
    publishedAt                 TIMESTAMP,
    createdAt                   TIMESTAMP,
    updatedAt                   TIMESTAMP,
    _created_at                 TIMESTAMP,
    _updated_at                 TIMESTAMP
)
USING DELTA
COMMENT 'Silver CMS news'
TBLPROPERTIES (
    'delta.enableChangeDataFeed'        = 'true',
    'delta.autoOptimize.optimizeWrite'  = 'true',
    'delta.autoOptimize.autoCompact'    = 'true',
    'layer'                             = 'silver',
    'domain'                            = 'content'
);

CREATE TABLE IF NOT EXISTS lh_vv_silver.dbo.cms_promotion_article (
    locale                      STRING      NOT NULL COMMENT 'Language code (locale)',
    id                          STRING      NOT NULL COMMENT 'Strapi numeric id cast to STRING',
    document_id                 STRING,
    slug                        STRING,
    title                       STRING,
    short_description           STRING,
    content_json                STRING,
    plain_text_content          STRING,
    terms_json                  STRING,
    plain_text_terms            STRING,
    poi_ids                     STRING,
    voucher_code                STRING,
    promotion_start_datetime    TIMESTAMP,
    promotion_end_datetime      TIMESTAMP,
    usage_start_datetime        TIMESTAMP,
    usage_end_datetime          TIMESTAMP,
    publishedAt                 TIMESTAMP,
    createdAt                   TIMESTAMP,
    updatedAt                   TIMESTAMP,
    _created_at                 TIMESTAMP,
    _updated_at                 TIMESTAMP
)
USING DELTA
COMMENT 'Silver CMS promotion article'
TBLPROPERTIES (
    'delta.enableChangeDataFeed'        = 'true',
    'delta.autoOptimize.optimizeWrite'  = 'true',
    'delta.autoOptimize.autoCompact'    = 'true',
    'layer'                             = 'silver',
    'domain'                            = 'content'
);


-- =============================================================================
-- CS-01  Bronze -> Silver: cms_destination
-- FIX: CROSS JOIN latest_batch -> chi lay MAX(crawled_at)
-- =============================================================================
WITH latest_batch AS (
    SELECT MAX(crawled_at) AS max_crawled_at
    FROM lh_vv_bronze.dbo.cms_raw_data
    WHERE source_name = 'strapi_destination'
),
parsed AS (
    SELECT
        b.source_name, b.type, b.event_id, b.crawled_at, b.ingested_date,
        from_json(b.raw_payload,
            'STRUCT<id:STRING,locale:STRING,name:STRING,slug:STRING,description:STRING,
             documentId:STRING,createdAt:STRING,updatedAt:STRING,publishedAt:STRING,
             cover_gallery:ARRAY<STRUCT<url:STRING,width:INT,height:INT>>,
             gallery:ARRAY<STRUCT<id:INT,locale:STRING,url:STRING,width:INT,height:INT,mime:STRING,provider:STRING>>,
             pois:ARRAY<STRING>>'
        ) AS p
    FROM lh_vv_bronze.dbo.cms_raw_data AS b
    CROSS JOIN latest_batch lb
    WHERE b.source_name = 'strapi_destination'
      AND b.raw_payload IS NOT NULL
      AND b.crawled_at  = lb.max_crawled_at
),
extracted AS (
    SELECT
        p.event_id, p.crawled_at,
        p.p.id          AS id,
        p.p.locale      AS locale,
        p.p.name        AS name,
        p.p.slug        AS slug,
        p.p.description AS description,
        p.p.documentId  AS document_id,
        p.p.createdAt   AS created_at_raw,
        p.p.updatedAt   AS updated_at_raw,
        p.p.publishedAt AS published_at_raw
    FROM parsed AS p
),
transformed AS (
    SELECT
        LOWER(COALESCE(e.locale, 'vi'))       AS language,
        CAST(e.id AS STRING)                  AS id,
        e.name                                AS name,
        e.slug                                AS name_normalized,
        e.description                         AS description,
        e.document_id                         AS documentId,
        CAST(e.created_at_raw  AS TIMESTAMP)  AS createdAt,
        CAST(e.updated_at_raw  AS TIMESTAMP)  AS updatedAt,
        CAST(e.published_at_raw AS TIMESTAMP) AS publishedAt,
        e.crawled_at                          AS _created_at,
        e.event_id                            AS event_id
    FROM extracted AS e
    WHERE e.id IS NOT NULL
),
deduped AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY documentId, language
                   ORDER BY updatedAt DESC NULLS LAST, _created_at DESC, event_id DESC
               ) AS rn
        FROM transformed
    ) t WHERE rn = 1
)
MERGE INTO lh_vv_silver.dbo.cms_destination AS tgt
USING (SELECT *, current_timestamp() AS _now FROM deduped) AS src
ON  tgt.documentId = src.documentId
AND tgt.language   = src.language
WHEN MATCHED AND (tgt.updatedAt < src.updatedAt OR tgt.updatedAt IS NULL) THEN UPDATE SET
    tgt.name            = src.name,
    tgt.name_normalized = src.name_normalized,
    tgt.description     = src.description,
    tgt.documentId      = src.documentId,
    tgt.createdAt       = src.createdAt,
    tgt.updatedAt       = src.updatedAt,
    tgt.publishedAt     = src.publishedAt,
    tgt._updated_at     = src._now
WHEN NOT MATCHED THEN INSERT
    (language, id, name, name_normalized, description, documentId, createdAt, updatedAt, publishedAt, _created_at, _updated_at)
VALUES
    (src.language, src.id, src.name, src.name_normalized, src.description, src.documentId, src.createdAt, src.updatedAt, src.publishedAt, src._created_at, src._now);


-- =============================================================================
-- CS-02  Bronze -> Silver: cms_blog
-- FIX: CROSS JOIN latest_batch -> chi lay MAX(crawled_at)
-- =============================================================================
WITH latest_batch AS (
    SELECT MAX(crawled_at) AS max_crawled_at
    FROM lh_vv_bronze.dbo.cms_raw_data
    WHERE source_name = 'strapi_blog'
),
parsed AS (
    SELECT b.event_id, b.crawled_at,
        from_json(b.raw_payload,
            'STRUCT<id:INT,documentId:STRING,locale:STRING,slug:STRING,title:STRING,sapo:STRING,
             author:STRING,first_publish_date:STRING,publishedAt:STRING,createdAt:STRING,updatedAt:STRING,
             destination:STRUCT<slug:STRING>,
             platforms:ARRAY<STRUCT<type:STRING>>,
             SEO:STRUCT<id:INT,seo_title:STRING,meta_description:STRING>,
             cover_image:STRUCT<url:STRING,hash:STRING,formats:STRUCT<
                 large:STRUCT<url:STRING>,medium:STRUCT<url:STRING>,
                 small:STRUCT<url:STRING>,thumbnail:STRUCT<url:STRING>>>,
             gallery:ARRAY<STRUCT<documentId:STRING,url:STRING,formats:STRUCT<
                 medium:STRUCT<url:STRING>,small:STRUCT<url:STRING>,thumbnail:STRUCT<url:STRING>>>>,
             pois:ARRAY<STRUCT<external_id:STRING>>,
             block_content:ARRAY<STRUCT<type:STRING,level:INT,format:STRING,language:STRING,
                 image:STRUCT<url:STRING,alternativeText:STRING,formats:STRUCT<
                     large:STRUCT<url:STRING>,medium:STRUCT<url:STRING>,
                     small:STRUCT<url:STRING>,thumbnail:STRUCT<url:STRING>>>,
                 children:ARRAY<STRUCT<type:STRING,text:STRING,bold:BOOLEAN,italic:BOOLEAN,code:BOOLEAN,
                     children:ARRAY<STRUCT<type:STRING,text:STRING,bold:BOOLEAN,italic:BOOLEAN>>>>>>>'
        ) AS p
    FROM lh_vv_bronze.dbo.cms_raw_data AS b
    CROSS JOIN latest_batch lb
    WHERE b.source_name = 'strapi_blog'
      AND b.raw_payload IS NOT NULL
      AND b.crawled_at  = lb.max_crawled_at
),
extracted AS (
    SELECT
        p.event_id, p.crawled_at,
        CAST(p.p.id AS STRING)                                  AS id,
        LOWER(COALESCE(p.p.locale, 'vi'))                       AS locale,
        p.p.documentId                                          AS document_id,
        p.p.slug                                                AS slug,
        p.p.title                                               AS title,
        p.p.sapo                                                AS sapo,
        p.p.author                                              AS author,
        to_json(p.p.block_content)                              AS block_content_json,
        CONCAT_WS('\n\n', FILTER(TRANSFORM(p.p.block_content, blk -> CASE blk.type
            WHEN 'paragraph' THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'heading'   THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'code'      THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'list'      THEN NULLIF(TRIM(CONCAT_WS('; ', FILTER(FLATTEN(TRANSFORM(blk.children, li ->
                CASE WHEN li.children IS NOT NULL
                     THEN FILTER(TRANSFORM(li.children, c -> COALESCE(c.text, '')), t -> trim(t) != '')
                     ELSE ARRAY() END
            )), t -> t IS NOT NULL AND trim(t) != ''))), '')
            ELSE NULL END), t -> t IS NOT NULL))                AS plain_text_body,
        p.p.cover_image.url                                     AS cover_image_url,
        p.p.cover_image.formats.medium.url                      AS cover_image_medium_url,
        p.p.cover_image.formats.small.url                       AS cover_image_small_url,
        p.p.cover_image.formats.thumbnail.url                   AS cover_image_thumbnail_url,
        to_json(TRANSFORM(p.p.gallery, g -> g.url))             AS gallery_urls,
        p.p.destination.slug                                    AS destination_slug,
        to_json(TRANSFORM(p.p.pois, poi -> poi.external_id))    AS poi_ids,
        to_json(TRANSFORM(p.p.platforms, pl -> pl.type))        AS platforms,
        p.p.SEO.seo_title                                       AS seo_title,
        p.p.SEO.meta_description                                AS seo_meta_description,
        TRY_CAST(p.p.first_publish_date AS DATE)                AS first_publish_date,
        TRY_CAST(p.p.publishedAt AS TIMESTAMP)                  AS publishedAt,
        TRY_CAST(p.p.createdAt   AS TIMESTAMP)                  AS createdAt,
        TRY_CAST(p.p.updatedAt   AS TIMESTAMP)                  AS updatedAt
    FROM parsed AS p
    WHERE p.p.id IS NOT NULL
),
deduped AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY document_id, locale
                   ORDER BY updatedAt DESC NULLS LAST, crawled_at DESC, event_id DESC
               ) AS rn
        FROM extracted
    ) t WHERE rn = 1
)
MERGE INTO lh_vv_silver.dbo.cms_blog AS tgt
USING (SELECT *, current_timestamp() AS _now FROM deduped) AS src
ON  tgt.document_id = src.document_id
AND tgt.locale      = src.locale
WHEN MATCHED AND (tgt.updatedAt < src.updatedAt OR tgt.updatedAt IS NULL) THEN UPDATE SET
    tgt.document_id               = src.document_id,
    tgt.slug                      = src.slug,
    tgt.title                     = src.title,
    tgt.sapo                      = src.sapo,
    tgt.author                    = src.author,
    tgt.block_content_json        = src.block_content_json,
    tgt.plain_text_body           = src.plain_text_body,
    tgt.cover_image_url           = src.cover_image_url,
    tgt.cover_image_medium_url    = src.cover_image_medium_url,
    tgt.cover_image_small_url     = src.cover_image_small_url,
    tgt.cover_image_thumbnail_url = src.cover_image_thumbnail_url,
    tgt.gallery_urls              = src.gallery_urls,
    tgt.destination_slug          = src.destination_slug,
    tgt.poi_ids                   = src.poi_ids,
    tgt.platforms                 = src.platforms,
    tgt.seo_title                 = src.seo_title,
    tgt.seo_meta_description      = src.seo_meta_description,
    tgt.first_publish_date        = src.first_publish_date,
    tgt.publishedAt               = src.publishedAt,
    tgt.createdAt                 = src.createdAt,
    tgt.updatedAt                 = src.updatedAt,
    tgt._updated_at               = src._now
WHEN NOT MATCHED THEN INSERT (
    locale, id, document_id, slug, title, sapo, author,
    block_content_json, plain_text_body,
    cover_image_url, cover_image_medium_url, cover_image_small_url, cover_image_thumbnail_url,
    gallery_urls, destination_slug, poi_ids, platforms,
    seo_title, seo_meta_description, first_publish_date,
    publishedAt, createdAt, updatedAt, _created_at, _updated_at
) VALUES (
    src.locale, src.id, src.document_id, src.slug, src.title, src.sapo, src.author,
    src.block_content_json, src.plain_text_body,
    src.cover_image_url, src.cover_image_medium_url, src.cover_image_small_url, src.cover_image_thumbnail_url,
    src.gallery_urls, src.destination_slug, src.poi_ids, src.platforms,
    src.seo_title, src.seo_meta_description, src.first_publish_date,
    src.publishedAt, src.createdAt, src.updatedAt, src.crawled_at, src._now
);


-- =============================================================================
-- CS-03  Bronze -> Silver: cms_news
-- FIX 1: CROSS JOIN latest_batch -> chi lay MAX(crawled_at)
-- FIX 2: WHEN MATCHED them guard updatedAt
-- =============================================================================
WITH latest_batch AS (
    SELECT MAX(crawled_at) AS max_crawled_at
    FROM lh_vv_bronze.dbo.cms_raw_data
    WHERE source_name = 'strapi_news'
),
parsed AS (
    SELECT b.event_id, b.crawled_at,
        from_json(b.raw_payload,
            'STRUCT<id:INT,documentId:STRING,locale:STRING,slug:STRING,title:STRING,sapo:STRING,
             is_featured:BOOLEAN,publishedAt:STRING,createdAt:STRING,updatedAt:STRING,
             news_category:STRUCT<category_name:STRING,key:STRING>,
             SEO:STRUCT<id:INT,seo_title:STRING,meta_description:STRING>,
             cover_image:STRUCT<url:STRING,hash:STRING,formats:STRUCT<
                 large:STRUCT<url:STRING>,medium:STRUCT<url:STRING>,
                 small:STRUCT<url:STRING>,thumbnail:STRUCT<url:STRING>>>,
             block_content:ARRAY<STRUCT<type:STRING,level:INT,format:STRING,language:STRING,
                 image:STRUCT<url:STRING,alternativeText:STRING,formats:STRUCT<
                     large:STRUCT<url:STRING>,medium:STRUCT<url:STRING>,
                     small:STRUCT<url:STRING>,thumbnail:STRUCT<url:STRING>>>,
                 children:ARRAY<STRUCT<type:STRING,text:STRING,bold:BOOLEAN,italic:BOOLEAN,code:BOOLEAN,
                     children:ARRAY<STRUCT<type:STRING,text:STRING,bold:BOOLEAN,italic:BOOLEAN>>>>>>>'
        ) AS p
    FROM lh_vv_bronze.dbo.cms_raw_data AS b
    CROSS JOIN latest_batch lb
    WHERE b.source_name = 'strapi_news'
      AND b.raw_payload IS NOT NULL
      AND b.crawled_at  = lb.max_crawled_at
),
extracted AS (
    SELECT
        p.event_id, p.crawled_at,
        CAST(p.p.id AS STRING)                                  AS id,
        LOWER(COALESCE(p.p.locale, 'vi'))                       AS locale,
        p.p.documentId                                          AS document_id,
        p.p.slug                                                AS slug,
        p.p.title                                               AS title,
        p.p.sapo                                                AS sapo,
        p.p.is_featured                                         AS is_featured,
        to_json(p.p.block_content)                              AS block_content_json,
        CONCAT_WS('\n\n', FILTER(TRANSFORM(p.p.block_content, blk -> CASE blk.type
            WHEN 'paragraph' THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'heading'   THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'code'      THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'list'      THEN NULLIF(TRIM(CONCAT_WS('; ', FILTER(FLATTEN(TRANSFORM(blk.children, li ->
                CASE WHEN li.children IS NOT NULL
                     THEN FILTER(TRANSFORM(li.children, c -> COALESCE(c.text, '')), t -> trim(t) != '')
                     ELSE ARRAY() END
            )), t -> t IS NOT NULL AND trim(t) != ''))), '')
            ELSE NULL END), t -> t IS NOT NULL))                AS plain_text_body,
        p.p.cover_image.url                                     AS cover_image_url,
        p.p.cover_image.formats.medium.url                      AS cover_image_medium_url,
        p.p.cover_image.formats.small.url                       AS cover_image_small_url,
        p.p.cover_image.formats.thumbnail.url                   AS cover_image_thumbnail_url,
        p.p.news_category.key                                   AS news_category_key,
        p.p.news_category.category_name                         AS news_category_name,
        p.p.SEO.seo_title                                       AS seo_title,
        p.p.SEO.meta_description                                AS seo_meta_description,
        TRY_CAST(p.p.publishedAt AS TIMESTAMP)                  AS publishedAt,
        TRY_CAST(p.p.createdAt   AS TIMESTAMP)                  AS createdAt,
        TRY_CAST(p.p.updatedAt   AS TIMESTAMP)                  AS updatedAt
    FROM parsed AS p
    WHERE p.p.id IS NOT NULL
),
deduped AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY document_id, locale
                   ORDER BY updatedAt DESC NULLS LAST, crawled_at DESC, event_id DESC
               ) AS rn
        FROM extracted
    ) t WHERE rn = 1
)
MERGE INTO lh_vv_silver.dbo.cms_news AS tgt
USING (SELECT *, current_timestamp() AS _now FROM deduped) AS src
ON  tgt.document_id = src.document_id
AND tgt.locale      = src.locale
WHEN MATCHED AND (tgt.updatedAt < src.updatedAt OR tgt.updatedAt IS NULL) THEN UPDATE SET
    tgt.document_id               = src.document_id,
    tgt.slug                      = src.slug,
    tgt.title                     = src.title,
    tgt.sapo                      = src.sapo,
    tgt.is_featured               = src.is_featured,
    tgt.block_content_json        = src.block_content_json,
    tgt.plain_text_body           = src.plain_text_body,
    tgt.cover_image_url           = src.cover_image_url,
    tgt.cover_image_medium_url    = src.cover_image_medium_url,
    tgt.cover_image_small_url     = src.cover_image_small_url,
    tgt.cover_image_thumbnail_url = src.cover_image_thumbnail_url,
    tgt.news_category_key         = src.news_category_key,
    tgt.news_category_name        = src.news_category_name,
    tgt.seo_title                 = src.seo_title,
    tgt.seo_meta_description      = src.seo_meta_description,
    tgt.publishedAt               = src.publishedAt,
    tgt.createdAt                 = src.createdAt,
    tgt.updatedAt                 = src.updatedAt,
    tgt._updated_at               = src._now
WHEN NOT MATCHED THEN INSERT (
    locale, id, document_id, slug, title, sapo, is_featured,
    block_content_json, plain_text_body,
    cover_image_url, cover_image_medium_url, cover_image_small_url, cover_image_thumbnail_url,
    news_category_key, news_category_name, seo_title, seo_meta_description,
    publishedAt, createdAt, updatedAt, _created_at, _updated_at
) VALUES (
    src.locale, src.id, src.document_id, src.slug, src.title, src.sapo, src.is_featured,
    src.block_content_json, src.plain_text_body,
    src.cover_image_url, src.cover_image_medium_url, src.cover_image_small_url, src.cover_image_thumbnail_url,
    src.news_category_key, src.news_category_name, src.seo_title, src.seo_meta_description,
    src.publishedAt, src.createdAt, src.updatedAt, src.crawled_at, src._now
);


-- =============================================================================
-- CS-04  Bronze -> Silver: cms_promotion_article
-- FIX: CROSS JOIN latest_batch -> chi lay MAX(crawled_at)
-- =============================================================================
WITH latest_batch AS (
    SELECT MAX(crawled_at) AS max_crawled_at
    FROM lh_vv_bronze.dbo.cms_raw_data
    WHERE source_name = 'strapi_ph_promotion_article'
),
parsed AS (
    SELECT b.event_id, b.crawled_at,
        from_json(b.raw_payload,
            'STRUCT<id:INT,documentId:STRING,locale:STRING,slug:STRING,title:STRING,
             shortDescription:STRING,voucherCode:STRING,
             promotionStartDateTime:STRING,promotionEndDateTime:STRING,
             usageStartDateTime:STRING,usageEndDateTime:STRING,
             publishedAt:STRING,createdAt:STRING,updatedAt:STRING,
             content:ARRAY<STRUCT<type:STRING,level:INT,format:STRING,language:STRING,
                 children:ARRAY<STRUCT<type:STRING,text:STRING,bold:BOOLEAN,italic:BOOLEAN,code:BOOLEAN,
                     children:ARRAY<STRUCT<type:STRING,text:STRING,bold:BOOLEAN,italic:BOOLEAN>>>>>>,
             termsAndConditions:ARRAY<STRUCT<type:STRING,level:INT,format:STRING,language:STRING,
                 children:ARRAY<STRUCT<type:STRING,text:STRING,bold:BOOLEAN,italic:BOOLEAN,code:BOOLEAN,
                     children:ARRAY<STRUCT<type:STRING,text:STRING,bold:BOOLEAN,italic:BOOLEAN>>>>>>,
             pois:ARRAY<STRUCT<external_id:STRING>>>'
        ) AS p
    FROM lh_vv_bronze.dbo.cms_raw_data AS b
    CROSS JOIN latest_batch lb
    WHERE b.source_name = 'strapi_ph_promotion_article'
      AND b.raw_payload IS NOT NULL
      AND b.crawled_at  = lb.max_crawled_at
),
extracted AS (
    SELECT
        p.event_id, p.crawled_at,
        CAST(p.p.id AS STRING)                                  AS id,
        LOWER(COALESCE(p.p.locale, 'vi'))                       AS locale,
        p.p.documentId                                          AS document_id,
        p.p.slug                                                AS slug,
        p.p.title                                               AS title,
        p.p.shortDescription                                    AS short_description,
        to_json(p.p.content)                                    AS content_json,
        CONCAT_WS('\n\n', FILTER(TRANSFORM(p.p.content, blk -> CASE blk.type
            WHEN 'paragraph' THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'heading'   THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'code'      THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'list'      THEN NULLIF(TRIM(CONCAT_WS('; ', FILTER(FLATTEN(TRANSFORM(blk.children, li ->
                CASE WHEN li.children IS NOT NULL
                     THEN FILTER(TRANSFORM(li.children, c -> COALESCE(c.text, '')), t -> trim(t) != '')
                     ELSE ARRAY() END
            )), t -> t IS NOT NULL AND trim(t) != ''))), '')
            ELSE NULL END), t -> t IS NOT NULL))                AS plain_text_content,
        to_json(p.p.termsAndConditions)                         AS terms_json,
        CONCAT_WS('\n\n', FILTER(TRANSFORM(p.p.termsAndConditions, blk -> CASE blk.type
            WHEN 'paragraph' THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'heading'   THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'code'      THEN NULLIF(TRIM(CONCAT_WS(' ', FILTER(TRANSFORM(blk.children, c -> COALESCE(c.text, '')), t -> trim(t) != ''))), '')
            WHEN 'list'      THEN NULLIF(TRIM(CONCAT_WS('; ', FILTER(FLATTEN(TRANSFORM(blk.children, li ->
                CASE WHEN li.children IS NOT NULL
                     THEN FILTER(TRANSFORM(li.children, c -> COALESCE(c.text, '')), t -> trim(t) != '')
                     ELSE ARRAY() END
            )), t -> t IS NOT NULL AND trim(t) != ''))), '')
            ELSE NULL END), t -> t IS NOT NULL))                AS plain_text_terms,
        to_json(TRANSFORM(p.p.pois, poi -> poi.external_id))    AS poi_ids,
        p.p.voucherCode                                         AS voucher_code,
        TRY_CAST(p.p.promotionStartDateTime AS TIMESTAMP)       AS promotion_start_datetime,
        TRY_CAST(p.p.promotionEndDateTime   AS TIMESTAMP)       AS promotion_end_datetime,
        TRY_CAST(p.p.usageStartDateTime     AS TIMESTAMP)       AS usage_start_datetime,
        TRY_CAST(p.p.usageEndDateTime       AS TIMESTAMP)       AS usage_end_datetime,
        TRY_CAST(p.p.publishedAt AS TIMESTAMP)                  AS publishedAt,
        TRY_CAST(p.p.createdAt   AS TIMESTAMP)                  AS createdAt,
        TRY_CAST(p.p.updatedAt   AS TIMESTAMP)                  AS updatedAt
    FROM parsed AS p
    WHERE p.p.id IS NOT NULL
),
deduped AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY document_id, locale
                   ORDER BY updatedAt DESC NULLS LAST, crawled_at DESC, event_id DESC
               ) AS rn
        FROM extracted
    ) t WHERE rn = 1
)
MERGE INTO lh_vv_silver.dbo.cms_promotion_article AS tgt
USING (SELECT *, current_timestamp() AS _now FROM deduped) AS src
ON  tgt.document_id = src.document_id
AND tgt.locale      = src.locale
WHEN MATCHED AND (tgt.updatedAt < src.updatedAt OR tgt.updatedAt IS NULL) THEN UPDATE SET
    tgt.document_id              = src.document_id,
    tgt.slug                     = src.slug,
    tgt.title                    = src.title,
    tgt.short_description        = src.short_description,
    tgt.content_json             = src.content_json,
    tgt.plain_text_content       = src.plain_text_content,
    tgt.terms_json               = src.terms_json,
    tgt.plain_text_terms         = src.plain_text_terms,
    tgt.poi_ids                  = src.poi_ids,
    tgt.voucher_code             = src.voucher_code,
    tgt.promotion_start_datetime = src.promotion_start_datetime,
    tgt.promotion_end_datetime   = src.promotion_end_datetime,
    tgt.usage_start_datetime     = src.usage_start_datetime,
    tgt.usage_end_datetime       = src.usage_end_datetime,
    tgt.publishedAt              = src.publishedAt,
    tgt.createdAt                = src.createdAt,
    tgt.updatedAt                = src.updatedAt,
    tgt._updated_at              = src._now
WHEN NOT MATCHED THEN INSERT (
    locale, id, document_id, slug, title, short_description,
    content_json, plain_text_content, terms_json, plain_text_terms,
    poi_ids, voucher_code, promotion_start_datetime, promotion_end_datetime,
    usage_start_datetime, usage_end_datetime,
    publishedAt, createdAt, updatedAt, _created_at, _updated_at
) VALUES (
    src.locale, src.id, src.document_id, src.slug, src.title, src.short_description,
    src.content_json, src.plain_text_content, src.terms_json, src.plain_text_terms,
    src.poi_ids, src.voucher_code, src.promotion_start_datetime, src.promotion_end_datetime,
    src.usage_start_datetime, src.usage_end_datetime,
    src.publishedAt, src.createdAt, src.updatedAt, src.crawled_at, src._now
);


-- =============================================================================
-- CG-00  Tao Gold Tables (DDL)
-- =============================================================================
CREATE TABLE IF NOT EXISTS lh_vv_gold.dbo.destination (
    id                STRING  COMMENT 'PK - Strapi documentId',
    lang              STRING  COMMENT 'Language code (locale)',
    code              STRING,
    name              STRING,
    name_normalized   STRING,
    parent_id         STRING,
    parent_code       STRING,
    path              STRING,
    short_description STRING,
    description       STRING,
    gallery           STRING,
    rating            STRING,
    statistics        STRING,
    reviews           STRING,
    ai_contents       STRING,
    is_active         BOOLEAN,
    metadata          STRING
)
USING DELTA
COMMENT 'Gold destination projection for serving and embedding pipeline.'
TBLPROPERTIES (
    'delta.enableChangeDataFeed'        = 'true',
    'delta.autoOptimize.optimizeWrite'  = 'true',
    'delta.autoOptimize.autoCompact'    = 'true',
    'layer'                             = 'gold',
    'domain'                            = 'content'
);

CREATE TABLE IF NOT EXISTS lh_vv_gold.dbo.blog (
    blog_id      STRING      NOT NULL,
    title        STRING      NOT NULL,
    sapo         STRING,
    body         STRING      NOT NULL,
    lang         STRING      NOT NULL,
    published_at TIMESTAMP   NOT NULL,
    updated_at   TIMESTAMP   NOT NULL
)
USING DELTA
COMMENT 'Gold blog projection for serving and embedding pipeline.'
TBLPROPERTIES ('layer'='gold','domain'='content');

CREATE TABLE IF NOT EXISTS lh_vv_gold.dbo.news (
    news_id      STRING      NOT NULL,
    title        STRING      NOT NULL,
    sapo         STRING,
    body         STRING      NOT NULL,
    lang         STRING      NOT NULL,
    category_key STRING,
    published_at TIMESTAMP   NOT NULL,
    updated_at   TIMESTAMP   NOT NULL
)
USING DELTA
COMMENT 'Gold news projection for serving and embedding pipeline.'
TBLPROPERTIES ('layer'='gold','domain'='content');


-- =============================================================================
-- CG-01  Silver -> Gold: blog
-- =============================================================================
WITH projected AS (
    SELECT
        s.document_id            AS blog_id,
        TRIM(s.title)            AS title,
        NULLIF(TRIM(s.sapo), '') AS sapo,
        s.plain_text_body        AS body,
        LOWER(s.locale)          AS lang,
        s.publishedAt            AS published_at,
        s.updatedAt              AS updated_at
    FROM lh_vv_silver.dbo.cms_blog AS s
    WHERE s.document_id     IS NOT NULL
      AND s.plain_text_body IS NOT NULL
      AND s.publishedAt     IS NOT NULL
      AND s.updatedAt       IS NOT NULL
)
MERGE INTO lh_vv_gold.dbo.blog AS tgt
USING projected AS src
ON  tgt.blog_id = src.blog_id
AND tgt.lang    = src.lang
WHEN MATCHED AND tgt.updated_at < src.updated_at THEN UPDATE SET
    tgt.title        = src.title,
    tgt.sapo         = src.sapo,
    tgt.body         = src.body,
    tgt.lang         = src.lang,
    tgt.published_at = src.published_at,
    tgt.updated_at   = src.updated_at
WHEN NOT MATCHED THEN INSERT (blog_id, title, sapo, body, lang, published_at, updated_at)
VALUES (src.blog_id, src.title, src.sapo, src.body, src.lang, src.published_at, src.updated_at);


-- =============================================================================
-- CG-02  Silver -> Gold: news
-- =============================================================================
WITH projected AS (
    SELECT
        s.document_id            AS news_id,
        TRIM(s.title)            AS title,
        NULLIF(TRIM(s.sapo), '') AS sapo,
        s.plain_text_body        AS body,
        LOWER(s.locale)          AS lang,
        s.news_category_key      AS category_key,
        s.publishedAt            AS published_at,
        s.updatedAt              AS updated_at
    FROM lh_vv_silver.dbo.cms_news AS s
    WHERE s.document_id     IS NOT NULL
      AND s.plain_text_body IS NOT NULL
      AND s.publishedAt     IS NOT NULL
      AND s.updatedAt       IS NOT NULL
)
MERGE INTO lh_vv_gold.dbo.news AS tgt
USING projected AS src
ON  tgt.news_id = src.news_id
AND tgt.lang    = src.lang
WHEN MATCHED AND tgt.updated_at < src.updated_at THEN UPDATE SET
    tgt.title        = src.title,
    tgt.sapo         = src.sapo,
    tgt.body         = src.body,
    tgt.lang         = src.lang,
    tgt.category_key = src.category_key,
    tgt.published_at = src.published_at,
    tgt.updated_at   = src.updated_at
WHEN NOT MATCHED THEN INSERT (news_id, title, sapo, body, lang, category_key, published_at, updated_at)
VALUES (src.news_id, src.title, src.sapo, src.body, src.lang, src.category_key, src.published_at, src.updated_at);


-- =============================================================================
-- CG-03  Silver -> Gold: destination
-- FIX: hierarchy CTE them ROW_NUMBER() -> chi lay 1 parent duy nhat / dest_id
--      tranh projected bi nhan doi truoc khi vao MERGE
-- =============================================================================
WITH dest_unique AS (
    SELECT DISTINCT dest_id, dest_name, parent_dest_id, parent_dest_name
    FROM lh_vv_gold.dbo.destination_ward_mapping
    WHERE dest_id IS NOT NULL
),
hierarchy AS (
    SELECT dest_id, parent_id FROM (
        SELECT
            d.dest_id,
            COALESCE(d.parent_dest_id, p.dest_id) AS parent_id,
            ROW_NUMBER() OVER (
                PARTITION BY d.dest_id
                ORDER BY p.dest_id DESC NULLS LAST
            ) AS rn
        FROM dest_unique AS d
        LEFT JOIN dest_unique AS p
            ON  p.dest_name != d.dest_name
            AND p.dest_name  = d.parent_dest_name
    ) t WHERE rn = 1
),
silver_latest AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY documentId, language
                   ORDER BY updatedAt DESC NULLS LAST, _updated_at DESC
               ) AS rn
        FROM lh_vv_silver.dbo.cms_destination
        WHERE documentId IS NOT NULL
    ) t WHERE rn = 1
),
projected AS (
    SELECT
        s.documentId         AS id,
        s.language           AS lang,
        s.name_normalized    AS code,
        s.name,
        s.name_normalized,
        h.parent_id,
        ps.name_normalized   AS parent_code,
        CASE
            WHEN h.parent_id IS NOT NULL THEN CONCAT(h.parent_id, '/', s.documentId)
            ELSE s.documentId
        END                  AS path,
        CAST(NULL AS STRING) AS short_description,
        s.description,
        CAST(NULL AS STRING) AS gallery,
        CAST(NULL AS STRING) AS rating,
        CAST(NULL AS STRING) AS statistics,
        CAST(NULL AS STRING) AS reviews,
        CAST(NULL AS STRING) AS ai_contents,
        (s.publishedAt IS NOT NULL) AS is_active,
        to_json(named_struct(
            'cms_created_at', s.createdAt,
            'cms_updated_at', s.updatedAt,
            'published_at',   s.publishedAt
        ))                   AS metadata
    FROM silver_latest AS s
    LEFT JOIN hierarchy     AS h  ON h.dest_id    = s.documentId
    LEFT JOIN silver_latest AS ps ON ps.documentId = h.parent_id
                                 AND ps.language   = s.language
)
MERGE INTO lh_vv_gold.dbo.destination AS tgt
USING projected AS src
ON  tgt.id   = src.id
AND tgt.lang = src.lang
WHEN MATCHED THEN UPDATE SET
    tgt.code            = src.code,
    tgt.name            = src.name,
    tgt.name_normalized = src.name_normalized,
    tgt.parent_id       = src.parent_id,
    tgt.parent_code     = src.parent_code,
    tgt.path            = src.path,
    tgt.description     = src.description,
    tgt.is_active       = src.is_active,
    tgt.metadata        = src.metadata
WHEN NOT MATCHED THEN INSERT (
    id, lang, code, name, name_normalized,
    parent_id, parent_code, path,
    short_description, description,
    gallery, rating, statistics, reviews, ai_contents,
    is_active, metadata
) VALUES (
    src.id, src.lang, src.code, src.name, src.name_normalized,
    src.parent_id, src.parent_code, src.path,
    src.short_description, src.description,
    src.gallery, src.rating, src.statistics, src.reviews, src.ai_contents,
    src.is_active, src.metadata
);

-- Kiem tra projected trong CG-03 co dup khong
WITH dest_unique AS (
    SELECT DISTINCT dest_id, dest_name, parent_dest_id, parent_dest_name
    FROM lh_vv_gold.dbo.destination_ward_mapping
    WHERE dest_id IS NOT NULL
),
hierarchy AS (
    SELECT
        d.dest_id,
        COALESCE(d.parent_dest_id, p.dest_id) AS parent_id,
        p.dest_name                             AS parent_name
    FROM dest_unique AS d
    LEFT JOIN dest_unique AS p
        ON  p.dest_name != d.dest_name
        AND p.dest_name  = d.parent_dest_name
),
silver_latest AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY documentId, language
                   ORDER BY updatedAt DESC NULLS LAST, _updated_at DESC
               ) AS rn
        FROM lh_vv_silver.dbo.cms_destination
        WHERE documentId IS NOT NULL
    ) t WHERE rn = 1
),
projected AS (
    SELECT
        s.documentId AS id,
        s.language   AS lang
    FROM silver_latest AS s
    LEFT JOIN hierarchy     AS h  ON h.dest_id    = s.documentId
    LEFT JOIN silver_latest AS ps ON ps.documentId = h.parent_id
                                 AND ps.language   = s.language
)
SELECT id, lang, COUNT(*) AS cnt
FROM projected
GROUP BY id, lang
HAVING COUNT(*) > 1
LIMIT 10;

-- =============================================================================
-- Step 1: lh_vv_silver.dbo.anchor_poi_map
-- =============================================================================

MERGE INTO lh_vv_silver.dbo.anchor_poi_map AS tgt

USING (
    SELECT
        name,
        MAX(documentId) AS documentId
    FROM lh_vv_silver.dbo.cms_destination
    -- WHERE name IN (
    --     'Huế',
    --     'Hội An',
    --     'Đà Nẵng',
    --     'Bà Nà Hills',
    --     'Phú Quốc'
    -- )
    GROUP BY name
) AS src

ON tgt.destination = src.name

WHEN MATCHED THEN
    UPDATE SET
        tgt.destination_id = src.documentId
;


-- =============================================================================
-- Step 2: lh_vv_gold.dbo.destination_ward_mapping
-- =============================================================================

MERGE INTO lh_vv_gold.dbo.destination_ward_mapping AS tgt

USING (
    SELECT
        name,
        MAX(documentId) AS documentId
    FROM lh_vv_silver.dbo.cms_destination
    -- WHERE name IN (
    --     'Huế',
    --     'Hội An',
    --     'Đà Nẵng',
    --     'Bà Nà Hills',
    --     'Phú Quốc'
    -- )
    GROUP BY name
) AS src

ON tgt.dest_name = src.name

WHEN MATCHED THEN
    UPDATE SET
        tgt.dest_id = src.documentId
;


-- =============================================================================
-- Validation
-- =============================================================================

SELECT
    'anchor_poi_map' AS tbl,
    destination,
    destination_id
FROM lh_vv_silver.dbo.anchor_poi_map
-- WHERE destination IN (
--     'Huế',
--     'Hội An',
--     'Đà Nẵng',
--     'Bà Nà Hills',
--     'Phú Quốc'
-- )
GROUP BY destination, destination_id;


SELECT
    'destination_ward_mapping' AS tbl,
    dest_name,
    dest_id
FROM lh_vv_gold.dbo.destination_ward_mapping
-- WHERE dest_name IN (
--     'Huế',
--     'Hội An',
--     'Đà Nẵng',
--     'Bà Nà Hills',
--     'Phú Quốc'
-- )
GROUP BY dest_name, dest_id;